In [1]:
import os
import re
import shutil
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from zhipuai_embedding import ZhipuAIEmbeddings
from load_doc import load_documents

# 加载环境变量
_ = load_dotenv(find_dotenv())

# ==========================================
# 1. 跨平台加载文档 (解决反斜杠路径隐患)
# ==========================================
print("⏳ 1. 开始加载文档...")
folder_path = Path("data_base")
loaders = []

# 使用 pathlib 的 rglob 遍历，自动处理 Linux/Windows 路径差异
documents = load_documents(Path("data_base"))

print(f"✅ 共加载 {len(documents)} 篇文档。")


# ==========================================
# 2. 智能清洗文本 (核心改进：保护代码格式)
# ==========================================
print("⏳ 2. 开始智能清洗文本...")

# 正则1：匹配“中文-换行-中文”，用于抹除中文语句被截断的换行
zh_pattern = re.compile(r'([\u4e00-\u9fff])\n([\u4e00-\u9fff])')
# 正则2：匹配“非中文-换行-非中文”，用于英文/代码，把换行变成空格防止单词粘连
en_pattern = re.compile(r'([^\u4e00-\u9fff])\n([^\u4e00-\u9fff])')

for doc in documents:
    source = doc.metadata.get('source', '').lower()
    content = doc.page_content
    
    if source.endswith('.pdf'):
        # 处理中文断句：直接缝合
        content = re.sub(zh_pattern, r'\1\2', content)
        # 处理英文断句：用空格衔接
        content = re.sub(en_pattern, r'\1 \2', content)
        # 移除无用的项目符号
        content = content.replace('•', '')
        
        # 🚨 警告：已删除全局 .replace('\n', '')，这样 SQL 语句的整体缩进和多行结构就能保住！
        
    elif source.endswith('.md'):
        # Markdown 通常会有太多无用空行，将其压缩，但绝不全删
        content = re.sub(r'\n{3,}', '\n\n', content)
        
    doc.page_content = content.strip()
print("✅ 文档清洗完成！")


# ==========================================
# 3. 切分文档
# ==========================================
print("⏳ 3. 开始切分文档...")
CHUNK_SIZE = 500
OVERLAP_SIZE = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE
)
split_docs = text_splitter.split_documents(documents)
print(f"✅ 切分完成，共产生 {len(split_docs)} 个文本块 (Chunks)。")


# ==========================================
# 4. 向量化入库 (核心改进：防脏数据残留)
# ==========================================
print("⏳ 4. 准备向量数据库...")
persist_directory = 'data_base/vector_db'

# 强制物理删除旧的数据库文件夹，确保每次存入的都是最新最干净的数据
if os.path.exists(persist_directory):
    print(f"🗑️ 发现旧数据库，正在清理旧数据...")
    shutil.rmtree(persist_directory, ignore_errors=True)

print("⏳ 开始生成 Embedding 并存入 Chroma (这可能需要一些时间，请耐心等待)...")
embedding = ZhipuAIEmbeddings()
vectordb = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding,
    persist_directory=persist_directory
)
print(f"✅ 向量库构建成功！当前库中共存储了 {vectordb._collection.count()} 个向量。")

d:\python\pyc\langchain project\.lang-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\23742\AppData\Local\Temp\ipykernel_32664\1021085166.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


⏳ 1. 开始加载文档...
✅ 共加载 834 篇文档。
⏳ 2. 开始智能清洗文本...
✅ 文档清洗完成！
⏳ 3. 开始切分文档...
✅ 切分完成，共产生 2939 个文本块 (Chunks)。
⏳ 4. 准备向量数据库...
🗑️ 发现旧数据库，正在清理旧数据...
⏳ 开始生成 Embedding 并存入 Chroma (这可能需要一些时间，请耐心等待)...
✅ 向量库构建成功！当前库中共存储了 2939 个向量。


In [5]:
# ==========================================
# 5. 检索测试
# ==========================================
print("\n🔍 5. 检索测试 (单纯的 Similarity 检测)")

question="OVER()怎么使用？"
     
sim_docs = vectordb.similarity_search(question,k=3)
print(f"检索到的内容数：{len(sim_docs)}")

for i, sim_doc in enumerate(sim_docs):
    print(f"检索到的第{i}个内容: \n{sim_doc.page_content[:200]}", end="\n--------------\n")


🔍 5. 检索测试 (单纯的 Similarity 检测)
检索到的内容数：3
检索到的第0个内容: 
OVER clause) will return the lowest and highest salar‐ ies in the table, respectively. The results are shown here: select ename,sal,        coalesce(lead(sal)over(order by sal),min(sal)over()) forward
--------------
检索到的第1个内容: 
over (order by sal desc) dr 5    from emp 6         ) x 7   where dr <= 5 The total number of rows returned may exceed five, but there will be only five dis‐ tinct salaries. Use ROW_NUMBER OVER if you
--------------
检索到的第2个内容: 
14 TURNER           30     14 WARD             30     14 The window function invocation in this example is COUNT(*) OVER(). The pres‐ ence of the OVER keyword indicates that the invocation of COUNT wi
--------------


In [6]:

print("\n🔍 5. 检索测试 (MMR 策略)")
question = "OVER()怎么使用？"
mmr_docs = vectordb.max_marginal_relevance_search(question, k=3)

for i, sim_doc in enumerate(mmr_docs):
    print(f"--- 结果 {i} ---")
    print(f"{sim_doc.page_content[:200]}...\n")


🔍 5. 检索测试 (MMR 策略)
--- 结果 0 ---
OVER clause) will return the lowest and highest salar‐ ies in the table, respectively. The results are shown here: select ename,sal,        coalesce(lead(sal)over(order by sal),min(sal)over()) forward...

--- 结果 1 ---
14 TURNER           30     14 WARD             30     14 The window function invocation in this example is COUNT(*) OVER(). The pres‐ ence of the OVER keyword indicates that the invocation of COUNT wi...

--- 结果 2 ---
28 vii...



In [4]:
# def main():
#     documents = load_documents(...)
#     documents = clean_documents(documents)
#     split_docs = split_documents(documents)
#     vectordb = build_vector_store(split_docs)
#     test_retrieval(vectordb)


# if __name__ == "__main__":
#     main()